# Notebook 06 — Retrieval Alignment: Novel Technique 3

**TIN-7: Cross-Lingual Embedding Alignment Analysis**

CKA measures geometric similarity, but does that similarity
**actually help** with a downstream task? This notebook tests
**functional alignment** via parallel sentence retrieval: given an
English sentence embedding, can we find its translation in another
language by nearest-neighbor search?

## Metrics

- **MRR (Mean Reciprocal Rank)**: Average of 1/rank. MRR=1.0 means
  every translation is the nearest neighbor.
- **Recall@1**: Fraction where the correct translation is the
  single nearest neighbor. The strictest test.
- **Recall@10**: Fraction where the translation is in the top 10.

In [ ]:
import logging
import sys
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from uth.utils.languages import Language
from uth.analysis.retrieval_metrics import (
    compute_all_retrieval_metrics,
    compute_cosine_similarity_matrix,
)
from uth.analysis.visualization import plot_retrieval_curves, plot_recall_bars

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

RESULTS_DIR = PROJECT_ROOT / "results" / "cross_lingual"
FIGURES_DIR = RESULTS_DIR / "figures"
METRICS_DIR = RESULTS_DIR / "metrics"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)

## 1. Load Cached Activations

In [ ]:
languages = list(Language)
language_names = [lang.lang_name for lang in languages]
iso_names = [lang.iso_code for lang in languages]
n_layers = 4

activations: dict[str, dict[str, torch.Tensor]] = {}
for lang in languages:
    activations[lang.lang_name] = {}
    for layer_idx in range(n_layers):
        filepath = RESULTS_DIR / "activations" / f"layer_{layer_idx}_{lang.lang_name}.pt"
        if filepath.exists():
            activations[lang.lang_name][f"layer_{layer_idx}"] = torch.load(
                filepath, weights_only=True
            )

print(f"Loaded activations for {len(activations)} languages.")

## 2. Compute Retrieval Metrics for All Layers

For each layer and each target language, we compute MRR, Recall@1,
Recall@5, and Recall@10 using English as the source language.

In [ ]:
SOURCE_LANG = "english"
K_VALUES = [1, 5, 10]

# Results: {layer_idx: {target_lang: {metric: value}}}
all_retrieval_results: dict[int, dict[str, dict[str, float]]] = {}

for layer_idx in range(n_layers):
    layer_name = f"layer_{layer_idx}"
    layer_results: dict[str, dict[str, float]] = {}

    src_emb = activations[SOURCE_LANG][layer_name].numpy()

    print(f"\n=== Layer {layer_idx} ===")
    for lang in languages:
        if lang.lang_name == SOURCE_LANG:
            continue

        tgt_emb = activations[lang.lang_name][layer_name].numpy()
        metrics = compute_all_retrieval_metrics(src_emb, tgt_emb, k_values=K_VALUES)
        layer_results[lang.lang_name] = metrics

        print(
            f"  {lang.iso_code:<4} MRR={metrics['mrr']:.4f}  "
            f"R@1={metrics['recall@1']:.4f}  "
            f"R@10={metrics['recall@10']:.4f}  "
            f"Median rank={metrics['median_rank']:.0f}"
        )

    all_retrieval_results[layer_idx] = layer_results

## 3. MRR vs. Layer Depth (Per Target Language)

In [ ]:
# Prepare data for plotting.
mrr_per_lang: dict[str, list[float]] = {}
for lang in languages:
    if lang.lang_name == SOURCE_LANG:
        continue
    mrr_values = []
    for layer_idx in range(n_layers):
        mrr_values.append(all_retrieval_results[layer_idx][lang.lang_name]["mrr"])
    mrr_per_lang[lang.iso_code] = mrr_values

fig = plot_retrieval_curves(
    layer_indices=list(range(n_layers)),
    mrr_per_layer=mrr_per_lang,
    title="Translation Retrieval MRR vs. Layer Depth\n(Source: English)",
    save_path=str(FIGURES_DIR / "retrieval_mrr_curve.png"),
)
plt.show()

## 4. Recall@1 and Recall@10 at Best Layer

Show per-language retrieval accuracy at the layer with the highest
average MRR.

In [ ]:
# Find the best layer (highest average MRR).
avg_mrr_per_layer = []
for layer_idx in range(n_layers):
    mrr_values = [
        all_retrieval_results[layer_idx][lang.lang_name]["mrr"]
        for lang in languages if lang.lang_name != SOURCE_LANG
    ]
    avg_mrr_per_layer.append(np.mean(mrr_values))

best_layer = int(np.argmax(avg_mrr_per_layer))
print(f"Best layer for retrieval: Layer {best_layer} (avg MRR = {avg_mrr_per_layer[best_layer]:.4f})")

# Recall@1 bar chart.
recall_at_1: dict[str, float] = {
    lang.iso_code: all_retrieval_results[best_layer][lang.lang_name]["recall@1"]
    for lang in languages if lang.lang_name != SOURCE_LANG
}

fig = plot_recall_bars(
    recall_scores=recall_at_1,
    k=1,
    layer_index=best_layer,
    save_path=str(FIGURES_DIR / f"recall_at_1_layer_{best_layer}.png"),
)
plt.show()

# Recall@10 bar chart.
recall_at_10: dict[str, float] = {
    lang.iso_code: all_retrieval_results[best_layer][lang.lang_name]["recall@10"]
    for lang in languages if lang.lang_name != SOURCE_LANG
}

fig = plot_recall_bars(
    recall_scores=recall_at_10,
    k=10,
    layer_index=best_layer,
    save_path=str(FIGURES_DIR / f"recall_at_10_layer_{best_layer}.png"),
)
plt.show()

## 5. Retrieval Error Analysis

For the best layer, show the hardest-to-retrieve sentence pairs —
cases where the correct translation is ranked furthest from the top.

In [ ]:
# Analyze retrieval errors for one target language.
ERROR_ANALYSIS_LANG = "arabic"
ERROR_ANALYSIS_LAYER = best_layer
layer_name = f"layer_{ERROR_ANALYSIS_LAYER}"

src_emb = activations[SOURCE_LANG][layer_name].numpy()
tgt_emb = activations[ERROR_ANALYSIS_LANG][layer_name].numpy()

# Compute full similarity matrix.
sim_matrix = compute_cosine_similarity_matrix(src_emb, tgt_emb)

# Find ranks of correct translations.
n_sentences = src_emb.shape[0]
ranks = np.zeros(n_sentences, dtype=np.int64)
for i in range(n_sentences):
    ranked = np.argsort(-sim_matrix[i])
    ranks[i] = np.where(ranked == i)[0][0] + 1

# Show worst-ranked translations (hardest to retrieve).
worst_indices = np.argsort(-ranks)[:10]

print(f"\n=== Hardest Retrievals: English -> {ERROR_ANALYSIS_LANG} (Layer {ERROR_ANALYSIS_LAYER}) ===")
print(f"{'Rank':>6} {'Sent#':>6}")
print("-" * 30)
for idx in worst_indices:
    print(f"  {ranks[idx]:>4}   #{idx}")

# Rank distribution histogram.
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(ranks, bins=50, color="#2196F3", edgecolor="white", alpha=0.8)
ax.set_xlabel("Rank of Correct Translation", fontsize=12)
ax.set_ylabel("Count", fontsize=12)
ax.set_title(
    f"Retrieval Rank Distribution: English -> {ERROR_ANALYSIS_LANG} (Layer {ERROR_ANALYSIS_LAYER})",
    fontsize=13, fontweight="bold",
)
ax.axvline(x=np.median(ranks), color="red", linestyle="--", label=f"Median={np.median(ranks):.0f}")
ax.legend()
plt.tight_layout()
fig.savefig(FIGURES_DIR / f"retrieval_rank_dist_{ERROR_ANALYSIS_LANG}.png", bbox_inches="tight")
plt.show()

## 6. Save Metrics

In [ ]:
retrieval_metrics: dict[str, float] = {}
for layer_idx in range(n_layers):
    for lang in languages:
        if lang.lang_name == SOURCE_LANG:
            continue
        metrics = all_retrieval_results[layer_idx][lang.lang_name]
        prefix = f"layer_{layer_idx}_en_{lang.iso_code}"
        retrieval_metrics[f"{prefix}_mrr"] = metrics["mrr"]
        retrieval_metrics[f"{prefix}_recall_at_1"] = metrics["recall@1"]
        retrieval_metrics[f"{prefix}_recall_at_10"] = metrics["recall@10"]

with open(METRICS_DIR / "retrieval_scores.json", "w") as f:
    json.dump(retrieval_metrics, f, indent=2)

print(f"Retrieval metrics saved to {METRICS_DIR / 'retrieval_scores.json'}")

## 7. Summary

**Key findings:**
- Best layer for translation retrieval: Layer [X].
- MRR ranges from [min] to [max] across target languages.
- [High/Low]-resource languages show [better/worse] retrieval.
- Functional alignment [does/does not] track geometric alignment.